In [ ]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\tashkil-dadgah-omoor-madani.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\preprocessed_data\tashkil-dadgah-omoor-madani_articles.tsv"

LAW_CODE = "tashkil_dadgah_omoor_madani"
LAW_NAME = "قانون آیین دادرسی دادگاه‌های عمومی و انقلاب در امور مدنی"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    return text.translate(str.maketrans(persian_digits, english_digits))


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    # حذف کاراکترهای کنترلی جهت‌دهنده
    text = text.replace("\u200F", "")  # Right-to-Left Mark
    text = text.replace("\u200E", "")  # Left-to-Right Mark (اختیاری)
    text = text.replace("\u200C", "")  # Zero Width Non-Joiner (اختیاری)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط فقط برای متادیتاهای معتبر
patterns_to_break = [
    r"(کتاب\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)[^\n]*)",
    r"(باب\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|\d+)[^\n]*)",
    r"(فصل\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|نخست|\d+)[^\n]*)",
    r"(مبحث\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)[^\n]*)",
    r"((?:ماده|مواد)\s+\d+(?:تا\d+)?(?:\s*و\s*\d+)?(?:مکرر)?(?:\([^)]+\))?\s*[-–ـ][^\n]*)"
]

for p in patterns_to_break:
    raw = re.sub(r"(?<!^)" + p, r"\n\1", raw, flags=re.MULTILINE | re.UNICODE)

lines = raw.splitlines()

# 3) پترن‌های دقیق
book_pattern = re.compile(
    r"^کتاب\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)\s*-\s*(.*)$"
)

bab_pattern = re.compile(
    r"^باب\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|\d+)\s*-\s*(.*)$"
)

chapter_pattern = re.compile(
    r"^فصل\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|چهاردهم|پانزدهم|نخست|\d+)\s*[-:]\s*(.*)$"
)

section_pattern = re.compile(
    r"^مبحث\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)\s*-\s*(.*)$"
)

# الگوی ماده با خط تیره اجباری
article_pattern = re.compile(
    r"^\s*(?:ماده|مواد)\s+"
    r"("
        r"\d+"
        r"(?:\s*تا\s*\d+)?"
        r"(?:\s*و\s*\d+)*"
    r")"
    r"(?:\s*مکرر)?"
    r"(?:\s*\([^)]+\))?\s*"
    r"[-–ـ]\s*(.*)$",    # ← خط تیره اجباری
    re.UNICODE
)

# متغیرهای سلسله‌مراتب
current_book = ""
current_bab = ""
current_chapter = ""
current_section = ""

articles = []
current_article_num = None
current_article_text = ""


def flush_article():
    global current_article_num, current_article_text
    if current_article_num is not None and current_article_text.strip():
        articles.append(
            {
                "law_code": LAW_CODE,
                "law_name": LAW_NAME,
                "book": current_book,
                "bab": current_bab,
                "chapter": current_chapter,
                "section": current_section,
                "article_number": current_article_num,
                "text": normalize_persian(current_article_text),
            }
        )
    current_article_num = None
    current_article_text = ""


# پردازش
for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # کتاب
    m = book_pattern.match(stripped)
    if m:
        flush_article()
        current_book = stripped
        current_bab = ""
        current_chapter = ""
        current_section = ""
        continue

    # باب
    m = bab_pattern.match(stripped)
    if m:
        flush_article()
        current_bab = stripped
        current_chapter = ""
        current_section = ""
        continue

    # فصل
    m = chapter_pattern.match(stripped)
    if m:
        flush_article()
        current_chapter = stripped
        current_section = ""
        continue

    # مبحث
    m = section_pattern.match(stripped)
    if m:
        flush_article()
        current_section = stripped
        continue

    # ماده
    m = article_pattern.match(stripped)
    if m:
        flush_article()
        num_str = m.group(1)
        rest = m.group(2).strip()

        # استخراج اولین عدد
        first_num = re.search(r"\d+", num_str)
        if first_num:
            try:
                current_article_num = int(first_num.group())
            except ValueError:
                current_article_num = None
        else:
            current_article_num = None

        current_article_text = f"ماده {num_str}- {rest}"
        continue

    # ادامه متن ماده
    if current_article_num is not None:
        current_article_text += " " + stripped

# آخرین ماده
flush_article()

# خروجی TSV
fieldnames = [
    "law_code", "law_name", "book", "bab", "chapter", "section", "article_number", "text"
]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش کامل شد.")
print("✓ تعداد مواد استخراج‌شده:", len(articles))
print("✓ مسیر خروجی:", output_tsv)


✓ پردازش کامل شد.
✓ تعداد مواد استخراج‌شده: 529
✓ مسیر خروجی: F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\preprocessed_data\tashkil-dadgah-omoor-madani_articles.tsv
